# VHS24K: Wedding VHS/DVD to 4K Restoration on Kaggle (2x NVIDIA T4)

## Before You Run
1. **Enable Dual T4 GPUs**: In the right-hand Kaggle sidebar under **Notebook options**:
   - Set **Accelerator** to **`GPU T4 x2`** (gives 2x 16GB T4 GPUs for parallel chunk upscaling).
   - Set **Internet** to **`On`** (required to install `Real-ESRGAN`, download model weights, and sync Google Drive).
2. **Connect Your Wedding Source Videos** (choose one):
   - **Option A — `rclone` (Recommended for private Google Drive files)**:
     1. Run `rclone config` locally on your machine to authorize a `gdrive` remote and copy your `rclone.conf`.
     2. Upload `rclone.conf` as a **Private Kaggle Dataset** (or paste it into `~/.config/rclone/rclone.conf`).
     3. Cell 4 will download `VTS_*.VOB` (recommended: un-recompressed DVD MPEG-2) or `Wedding*.mp4` from `gdrive:Wedding`.
   - **Option B — Private Kaggle Dataset**:
     Upload your `VTS_*.VOB` / `Wedding*.mp4` files to a private Kaggle Dataset and point `INPUT_DIR` in Cell 4 to `/kaggle/input/<your-dataset-name>`.
3. **Resumable 12-Hour Sessions**:
   `VHS24K` tracks every 60-second chunk in `work/<source>/manifest.json` using atomic writes. If a Kaggle session hits the 12-hour limit, re-running Cell 6 will skip already-completed chunks and resume where it left off.

In [ ]:
!nvidia-smi
!ffmpeg -version | head -n 3

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/kaggle/working")
VHS24K_DIR = WORK_ROOT / "VHS24K"
REALESRGAN_DIR = WORK_ROOT / "Real-ESRGAN"

# 1. Clone or pull latest VHS24K and install
if VHS24K_DIR.exists():
    !git -C {VHS24K_DIR} pull
else:
    !git clone https://github.com/claytonjfaria-ai/VHS24K.git {VHS24K_DIR}
!pip install -q -e {VHS24K_DIR}
import sys, importlib
if str(VHS24K_DIR) not in sys.path:
    sys.path.insert(0, str(VHS24K_DIR))
importlib.invalidate_caches()

# 2. Install patched BasicSR, Real-ESRGAN, GFPGAN, and model weights
from vhs24k.kaggle_setup import setup_kaggle_environment
setup_kaggle_environment(REALESRGAN_DIR)

In [ ]:
from pathlib import Path
from vhs24k.drive_io import download_wedding_files

INPUT_DIR = Path("/kaggle/working/input")
WORK_DIR = Path("/kaggle/working/work")
OUTPUT_DIR = Path("/kaggle/working/output")
SOURCE_MODE = "vob"  # "vob" (recommended un-recompressed DVD) or "mp4"

# Option A: Download from Google Drive via rclone (requires ~/.config/rclone/rclone.conf)
# Install rclone on Kaggle if missing:
!which rclone || (curl -s https://rclone.org/install.sh | sudo bash)

# Copy your rclone.conf from a private Kaggle Dataset if attached, e.g.:
# !mkdir -p ~/.config/rclone && cp /kaggle/input/rclone-config/rclone.conf ~/.config/rclone/rclone.conf

downloaded = download_wedding_files(
    dest_dir=INPUT_DIR,
    source_type=SOURCE_MODE,
    method="rclone",          # or "gdown" if files are link-shared
    rclone_remote="gdrive:Wedding",
)
print("Downloaded files:", [p.name for p in downloaded])

# Option B: If using a Kaggle Private Dataset instead of rclone, set:
# INPUT_DIR = Path("/kaggle/input/wedding-vhs-sources")
!python -m vhs24k.cli inspect --input-dir {INPUT_DIR} --source-mode {SOURCE_MODE}

In [ ]:
# Run a 15-second test clip first to verify 4K framing, deinterlacing, and face enhancement
!python -m vhs24k.cli run \
    --input-dir {INPUT_DIR} \
    --work-dir /kaggle/working/work_test \
    --output-dir {OUTPUT_DIR} \
    --source-mode {SOURCE_MODE} \
    --model realesr-general-x4v3 \
    --4k-mode pillarbox_uhd \
    --num-gpus 2 \
    --test-clip 15 \
    --realesrgan-dir /kaggle/working/Real-ESRGAN

!ls -lh {OUTPUT_DIR}

In [ ]:
# Full resumable dual-T4 pipeline (safe to re-run if interrupted at 12h)
!python -m vhs24k.cli run \
    --input-dir {INPUT_DIR} \
    --work-dir {WORK_DIR} \
    --output-dir {OUTPUT_DIR} \
    --source-mode {SOURCE_MODE} \
    --model realesr-general-x4v3 \
    --chunk-seconds 60 \
    --4k-mode pillarbox_uhd \
    --num-gpus 2 \
    --realesrgan-dir /kaggle/working/Real-ESRGAN

# Sync finished 4K MP4s back to Google Drive
from vhs24k.drive_io import upload_outputs
upload_outputs(OUTPUT_DIR, rclone_remote="gdrive:Wedding/4K_Output")
!ls -lh {OUTPUT_DIR}